In [1]:
import os
import sys
import json
project_dir = os.path.dirname(os.getcwd())
sys.path.append(project_dir)

import numpy as np  # Ensure this is imported if not already


import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'
seed = torch.Generator().manual_seed(42)
print(device)

cuda


# Data

In [2]:
from data.cifar10 import get_cifar10_pipeline

train_loader, val_loader, test_loader = get_cifar10_pipeline(batch_size=128, indexed=True)
sample_x, sample_y, idx = next(iter(train_loader))
print(sample_x.shape)
print(sample_y.shape)
print(idx.shape)

Files already downloaded and verified
Files already downloaded and verified
torch.Size([128, 3, 32, 32])
torch.Size([128])
torch.Size([128])


# Self Distill

In [3]:
import tqdm
from utils.train import evaluate_model
from utils.losses import DistillationLoss, Accuracy


def self_distill_model(train_loader, model, criterion, optimizer, scheduler, device, teacher_logits_dict):
    model.train()
    epoch_loss = []

    for inputs, labels, indices in tqdm.tqdm(train_loader, desc='training...', file=sys.stdout):
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)

        # Determine if we can apply self-distillation
        if all(idx.item() in teacher_logits_dict for idx in indices):
            prev_logits = torch.stack([teacher_logits_dict[idx.item()] for idx in indices]).to(device)
            loss = criterion(outputs, prev_logits.detach(), labels)
        else:
            loss = F.cross_entropy(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Store current logits for self-distillation in next epoch
        for i, idx in enumerate(indices):
            teacher_logits_dict[idx.item()] = outputs[i].detach().clone()

        epoch_loss.append(loss.item())

    scheduler.step()
    return np.mean(epoch_loss)


def distill_val(train_loader, val_loader, model, criterion, optimizer, scheduler, 
                device='cpu', aux_metrics={}, path="./temp.pth", patience=50, epochs=50):
    metrics = {"train_loss": [], "accuracy": []}
    for k in aux_metrics.keys():
        metrics[k] = []

    teacher_logits_dict = {}
    best_val_acc = 0
    counter = 0
    for epoch in range(epochs):
        train_loss = self_distill_model(train_loader, model, criterion, optimizer, scheduler, device, teacher_logits_dict)
        val_acc = evaluate_model(val_loader, model, Accuracy(), device)
        metrics['train_loss'].append(train_loss)
        metrics['accuracy'].append(val_acc)
        for k, v in aux_metrics.items():
            metrics[k].append(evaluate_model(val_loader, model, v, device))
        if metrics['accuracy'][-1] >= best_val_acc:
            best_val_acc = metrics['accuracy'][-1]
            counter = 0
            print(f"Epoch {epoch+1}: New best accuracy: {best_val_acc:.4f} saving model...")
            state = {
                'epoch': epoch,
                'state_dict': model.state_dict(),
                'optimizer': optimizer.state_dict()
            }
            state.update({k: metrics[k][-1] for k in metrics.keys()})
            torch.save(state, path)
        else:
            counter += 1
        if counter >= patience:
            print(f"Epoch {epoch+1}: Early stop triggered.")
            break
    return metrics

In [ ]:
from models.resnet import resnet34, resnet50
from utils.plots import plot_training_metrics
from utils.losses import DistillationLoss
import torch.optim as optim

model = resnet50().to(device)
criterion = DistillationLoss(T=3.0, alpha=0.7)
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.1)
path = f'../models/weights/cifar10_resnet50_self.pth'
metrics = distill_val(train_loader, val_loader, model, criterion, optimizer, scheduler, 
                      device=device, path=path, patience=50, epochs=50)
plot_training_metrics(metrics)
with open(os.path.join(project_dir, 'data/states/cifar10_resnet50_self.json'), 'w') as f:
    json.dump(metrics, f)

training...:   0%|          | 0/391 [00:00<?, ?it/s]